<a href="https://colab.research.google.com/github/crunchdao/quickstarters/blob/master/competitions/mid-one/momentum_attacker/momentum_attacker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%env API_BASE_URL=https://api.hub.crunchdao.io

env: API_BASE_URL=https://api.hub.crunchdao.io


# Momentum Attacker
This notebook demonstrates how to create an `Attacker` described in [attacker.md](https://github.com/microprediction/endersgame/blob/main/endersgame/attackers/attacker.md). You may want to glance at this [notebook](https://github.com/microprediction/endersnotebooks/blob/main/mean_reversion_attacker.ipynb) also, if you seek more context or wish to know how these attackers can be used in a new tournament.



## Setup

In [ ]:
%pip install --upgrade endersgame

In [ ]:
%pip install --upgrade crunch-cli
!crunch setup --notebook enders hello --token aaaabbbbccccddddeeeeffff

## Imports

In [23]:
import math
import typing

import pandas
from endersgame import HORIZON, Attacker, FEWMean, FEWVar
from endersgame.accounting.pnlutil import add_pnl_summaries, zero_pnl_summary
from tqdm.auto import tqdm

In [3]:
import crunch

crunch = crunch.load_notebook()

loaded inline runner with module: <module '__main__'>


## Creating a Momentum based Attacker

We derive from `Attacker` and use the utilities `FEWMean` and `FEWVar` to track the running exponentially weighted quantities we need.

In [10]:

class MyAttacker(Attacker):

    def __init__(
        self,
        fast_fading_factor=0.1,
        slow_fading_factor=0.01,
        diff_fading_factor=0.001,
        threshold=2,
        burn_in=100,
        **kwargs
    ):
        super().__init__(**kwargs)

        # Track fast expon-weighted moving average
        self.fast_ewa = FEWMean(fading_factor=fast_fading_factor)

        # Track slow expon-weighted moving average
        self.slow_ewa = FEWMean(fading_factor=slow_fading_factor)

        # Tracks mean and var of the difference between the two
        self.diff_var = FEWVar(fading_factor=diff_fading_factor)

        self.threshold = threshold
        self.countdown = burn_in

    def tick(self, x: float):
        # Update the fast expon avg
        self.fast_ewa.tick(x=x)

        # Update the slow expon avg
        self.slow_ewa.tick(x=x)

        fast_minus_slow = self.fast_ewa.get() - self.slow_ewa.get()
        #  Update var of diff
        self.diff_var.tick(x=fast_minus_slow)

        # Soon we'll be warm
        self.countdown -= 1

    def predict(self, horizon=HORIZON) -> float:
        """
        We buy if signal > threshold*(trailing std of signal)
        """

        if self.countdown > 0:
            return 0  # Not warmed up

        fast_minus_slow = self.fast_ewa.get() - self.slow_ewa.get()
        try:
            fast_minus_slow_std = math.sqrt(self.diff_var.get())
            decision = int(fast_minus_slow / (self.threshold * fast_minus_slow_std))  # <--- Create a buy (>0) or sell (<0) decision
            return decision
        except ArithmeticError:
            return 0

## Run the attacker on mock data

We use `tick_and_predict` from the parent class as this will track profit and loss for us.

In [11]:
# Always reset an attacker
attacker = MyAttacker()

data = [1, 3, 4, 2, 4, 5, 1, 5, 2, 5, 10] * 100
for x in data:
    y = attacker.tick_and_predict(x=x)

## Run the attacker on real data

We reset the attacker every time it encounters a new stream, but track aggregate statistics.

In [ ]:
x_train, x_test = crunch.load_streams()

In [19]:
total_pnl = []

for stream in tqdm(x_train):
    attacker = MyAttacker()
    pnl = zero_pnl_summary()

    for message in tqdm(stream, leave=False):
        attacker.tick_and_predict(x=message['x'])

    stream_pnl = attacker.pnl.summary()

    pnl = add_pnl_summaries(pnl, stream_pnl)
    pnl.update({
        'profit_per_decision': pnl['total_profit'] / pnl['num_resolved_decisions']
    })

    total_pnl.append(pnl)

total_pnl = pandas.DataFrame(total_pnl)

  0%|          | 0/80 [00:00<?, ?it/s]

  0%|          | 0/4892 [00:00<?, ?it/s]

  0%|          | 0/18886 [00:00<?, ?it/s]

  0%|          | 0/14472 [00:00<?, ?it/s]

  0%|          | 0/4396 [00:00<?, ?it/s]

...

  0%|          | 0/3585 [00:00<?, ?it/s]

  0%|          | 0/4299 [00:00<?, ?it/s]

  0%|          | 0/17312 [00:00<?, ?it/s]

  0%|          | 0/13501 [00:00<?, ?it/s]

  0%|          | 0/3874 [00:00<?, ?it/s]

In [20]:
total_pnl

,total_profit,num_resolved_decisions,wins,losses,current_ndx,profit_per_decision
0,2.308571,362,153,209,4892,0.006377
1,-261.415714,1013,383,630,18886,-0.258061
2,255.349048,777,462,315,14472,0.328635
3,-24.694286,148,39,109,4396,-0.166853
4,-63.636667,457,153,304,4777,-0.139249
...,...,...,...,...,...,...
75,-72.680000,428,149,279,3585,-0.169813
76,-17.027046,611,218,393,4299,-0.027868
77,61.693526,1270,707,563,17312,0.048578
78,-109.772529,877,438,439,13501,-0.125168


## CrunchDAO Code Interface

[Submitting to the CrunchDAO platform requires 2 functions, `train` and `infer`.](https://docs.crunchdao.com/competitions/code-interface) Any line that is not in a function or is not an import will be commented when the notebook is processed.

The content of the function is the same as the example, but the train must save the model to be read in infer.

In [21]:
def train():
    pass  # no train

In [25]:
def infer(
    stream: typing.Iterator[dict],
):
    attacker = MyAttacker()
    total_pnl = zero_pnl_summary()

    yield  # mark as ready

    for message in stream:
        yield attacker.tick_and_predict(x=message['x'])

    stream_pnl = attacker.pnl.summary()
    total_pnl = add_pnl_summaries(total_pnl, stream_pnl)

    total_pnl.update({
        'profit_per_decision': total_pnl['total_profit'] / total_pnl['num_resolved_decisions']
    })

    print(total_pnl)

In [26]:
crunch.test()

print("Download this notebook and submit it to the platform: https://hub.crunchdao.com/competitions/endersgame/submit/via/notebook")

18:24:35 no forbidden library found
18:24:35 
18:24:35 started
18:24:35 running local test
18:24:35 internet access isn't restricted, no check will be done
18:24:35 
18:24:36 starting stream loop...
18:24:36 call: train - stream.len=80


download data\X_train.parquet from https:crunchdao--competition--staging.s3.eu-west-1.amazonaws.com/data-releases/72/X_train.parquet (2248900 bytes)
already exists: file length match
download data\y_train.parquet from https:crunchdao--competition--staging.s3.eu-west-1.amazonaws.com/data-releases/72/y_train.parquet (1828809 bytes)
already exists: file length match
download data\X_test.parquet from https:crunchdao--competition--staging.s3.eu-west-1.amazonaws.com/data-releases/72/X_test_reduced.parquet (141137 bytes)
already exists: file length match
download data\y_test.parquet from https:crunchdao--competition--staging.s3.eu-west-1.amazonaws.com/data-releases/72/y_test_reduced.parquet (145664 bytes)
already exists: file length match
download data\example_prediction.parquet from https:crunchdao--competition--staging.s3.eu-west-1.amazonaws.com/data-releases/72/example_prediction_reduced.parquet (145664 bytes)
already exists: file length match


18:24:36 looping stream=`aud-jpy` (1/20)
18:24:36 call: infer (1/1)
18:24:36 looping stream=`aud-nzd` (2/20)
18:24:36 call: infer (1/1)
18:24:36 looping stream=`aud-usd` (3/20)
18:24:36 call: infer (1/1)
18:24:36 looping stream=`eur-aud` (4/20)
18:24:36 call: infer (1/1)
18:24:36 looping stream=`eur-chf` (5/20)
18:24:36 call: infer (1/1)
18:24:36 looping stream=`eur-gbp` (6/20)
18:24:36 call: infer (1/1)
18:24:37 looping stream=`eur-jpy` (7/20)
18:24:37 call: infer (1/1)
18:24:37 looping stream=`eur-nok` (8/20)
18:24:37 call: infer (1/1)
18:24:37 looping stream=`eur-usd` (9/20)
18:24:37 call: infer (1/1)
18:24:37 looping stream=`gbp-aud` (10/20)
18:24:37 call: infer (1/1)
18:24:37 looping stream=`gbp-jpy` (11/20)


{'total_profit': -29.84500000000533, 'num_resolved_decisions': 227, 'wins': 80, 'losses': 147, 'current_ndx': 1987, 'profit_per_decision': -0.13147577092513363}
{'total_profit': -51.308571428565976, 'num_resolved_decisions': 263, 'wins': 114, 'losses': 149, 'current_ndx': 1859, 'profit_per_decision': -0.1950896252036729}
{'total_profit': -259.02571428565017, 'num_resolved_decisions': 324, 'wins': 112, 'losses': 212, 'current_ndx': 1647, 'profit_per_decision': -0.7994620811285499}
{'total_profit': 272.72736842120725, 'num_resolved_decisions': 222, 'wins': 137, 'losses': 85, 'current_ndx': 1967, 'profit_per_decision': 1.2285016595549876}
{'total_profit': 256.069999999987, 'num_resolved_decisions': 243, 'wins': 174, 'losses': 69, 'current_ndx': 1839, 'profit_per_decision': 1.0537860082303991}
{'total_profit': 460.0399999995419, 'num_resolved_decisions': 271, 'wins': 190, 'losses': 81, 'current_ndx': 1560, 'profit_per_decision': 1.697564575644066}
{'total_profit': 300.59909090919416, 'num_

18:24:37 call: infer (1/1)
18:24:37 looping stream=`gbp-usd` (12/20)
18:24:37 call: infer (1/1)
18:24:37 looping stream=`nzd-jpy` (13/20)
18:24:37 call: infer (1/1)
18:24:37 looping stream=`nzd-usd` (14/20)
18:24:37 call: infer (1/1)
18:24:37 looping stream=`usd-cad` (15/20)
18:24:37 call: infer (1/1)
18:24:37 looping stream=`usd-chf` (16/20)
18:24:37 call: infer (1/1)
18:24:37 looping stream=`usd-jpy` (17/20)
18:24:37 call: infer (1/1)
18:24:37 looping stream=`usd-mxn` (18/20)
18:24:37 call: infer (1/1)
18:24:37 looping stream=`usd-sgd` (19/20)
18:24:37 call: infer (1/1)
18:24:37 looping stream=`usd-zar` (20/20)
18:24:37 call: infer (1/1)


{'total_profit': -44.33176470590554, 'num_resolved_decisions': 242, 'wins': 101, 'losses': 141, 'current_ndx': 2052, 'profit_per_decision': -0.1831891103549816}
{'total_profit': 41.603333333386665, 'num_resolved_decisions': 223, 'wins': 130, 'losses': 93, 'current_ndx': 1812, 'profit_per_decision': 0.186562032885142}
{'total_profit': -54.32956521741099, 'num_resolved_decisions': 296, 'wins': 114, 'losses': 182, 'current_ndx': 2023, 'profit_per_decision': -0.18354582843719927}
{'total_profit': -77.79666666662891, 'num_resolved_decisions': 413, 'wins': 189, 'losses': 224, 'current_ndx': 1660, 'profit_per_decision': -0.18836965294583272}
{'total_profit': 123.40000000001291, 'num_resolved_decisions': 205, 'wins': 107, 'losses': 98, 'current_ndx': 1634, 'profit_per_decision': 0.601951219512258}
{'total_profit': -315.5044444443353, 'num_resolved_decisions': 356, 'wins': 133, 'losses': 223, 'current_ndx': 1782, 'profit_per_decision': -0.8862484394503801}
{'total_profit': -362.82000000033173, 

18:24:37 save prediction - path=data\prediction.csv


{'total_profit': -18.552162162165658, 'num_resolved_decisions': 221, 'wins': 106, 'losses': 115, 'current_ndx': 1659, 'profit_per_decision': -0.08394643512292153}


18:24:37 prediction is valid
18:24:37 ended
18:24:37 duration - time=00:00:02
18:24:37 memory - before="238.52 MB" after="275.19 MB" consumed="36.67 MB"


Download this notebook and submit it to the platform: https://hub.crunchdao.com/competitions/endersgame/submit/via/notebook
